In [10]:
import os
from pathlib import Path

if Path.cwd().name != "book-text-ml-improved":
    for candidate in [Path("book-text-ml-improved"), *Path(".").glob("*/book-text-ml-improved")]:
        if candidate.is_dir():
            os.chdir(candidate)
            break

print("작업 폴더:", Path.cwd())

작업 폴더: c:\dev\llm-data-analysis-course\notebooks\book-text-ml-improved


In [11]:
import pandas as pd

df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")
print(df.shape)

(593, 13)


In [12]:
from kiwipiepy import Kiwi

kiwi = Kiwi()
TARGET_TAGS = {"NNG", "NNP", "SL"}

def extract_tokens(text):
    tokens = kiwi.tokenize(str(text))
    return " ".join(t.form for t in tokens if t.tag in TARGET_TAGS)

df["상품명_토큰"] = df["상품명"].apply(extract_tokens)

In [13]:
# [1단계] 가장 단순한 방법 - 현재 폴더 기준으로 경로를 만든다.
# 노트북이 notebooks/ch04 안에서 실행되면 엉뚱한 경로가 나온다는 것을 확인하는 셀이다.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print(DATA_DIR)
print("이 경로가 실제로 존재하는가?", DATA_DIR.exists())


c:\dev\llm-data-analysis-course\notebooks\book-text-ml-improved\data\raw
이 경로가 실제로 존재하는가? False


In [14]:
# [2단계] 위로 올라가며 data 폴더를 직접 찾는다.
current = Path.cwd().resolve()
project_root = current
print("현재 위치:", current)

while True:
    # 지금 폴더 안에 data 폴더가 있으면 여기가 프로젝트 루트다.
    if (current / "data").is_dir():
        project_root = current
        break

    # 최상위까지 올라왔는데도 못 찾았다면 오류를 낸다.
    # (C:\ 의 부모는 C:\ 자기 자신이라, 이 검사가 없으면 무한 루프가 된다.)
    if current == current.parent:
        raise FileNotFoundError("data 폴더를 찾지 못했습니다.")

    # 못 찾았으면 부모 폴더로 한 단계 올라가서 다시 확인한다.
    current = current.parent

print("프로젝트 루트:", project_root)


현재 위치: C:\dev\llm-data-analysis-course\notebooks\book-text-ml-improved
프로젝트 루트: C:\dev\llm-data-analysis-course


In [15]:
# [3단계] 같은 일을 하는 함수로 정리한다.
from pathlib import Path


def get_project_root():
    """현재 폴더에서 위로 올라가며 data 폴더를 가진 폴더를 찾아 반환한다."""
    current = Path.cwd().resolve()

    while True:
        if (current / "data").is_dir():
            return current

        # 최상위까지 올라왔는데도 못 찾았으면 오류를 낸다 (무한 루프 방지)
        if current == current.parent:
            raise FileNotFoundError("data 폴더를 찾지 못했습니다.")

        current = current.parent


def get_data_dir():
    """프로젝트 루트 아래의 data 폴더 경로를 반환한다."""
    return get_project_root() / "data"


print("프로젝트 루트:", get_project_root())
print("데이터 폴더:", get_data_dir())


프로젝트 루트: C:\dev\llm-data-analysis-course
데이터 폴더: C:\dev\llm-data-analysis-course\data


In [16]:
# [4단계] course_utils 를 import 할 수 있도록 프로젝트 루트를 sys.path 에 넣는다.
from pathlib import Path
import sys

# ---- 프로젝트 루트 찾기 (노트북을 어느 폴더에서 열어도 동작한다) ----
# 현재 폴더에서 시작해 위로 한 단계씩 올라가며 data 폴더가 있는 곳을 찾는다.
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "data").is_dir():
    # 최상위(C:\)까지 올라왔는데도 못 찾으면 멈춘다. 이 검사가 없으면 무한 루프가 된다.
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("data 폴더를 찾지 못했습니다.")
    PROJECT_ROOT = PROJECT_ROOT.parent

# course_utils 를 import 할 수 있도록 프로젝트 루트를 검색 경로에 넣는다.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR, "| 존재:", DATA_DIR.exists())


프로젝트 루트: C:\dev\llm-data-analysis-course
데이터 폴더: C:\dev\llm-data-analysis-course\data\raw | 존재: True


In [17]:
from course_utils.paths import get_project_root, get_data_dir

print("프로젝트 루트 :", get_project_root())
print("데이터 폴더 :", get_data_dir())


프로젝트 루트 : C:\dev\llm-data-analysis-course
데이터 폴더 : C:\dev\llm-data-analysis-course\data


In [18]:
# 1) 최소한의 불용어만 지정 (분야 핵심 단어는 넣지 않는다)
stopwords = {
    "에디션",
}

# 2) 형태소 분석 결과(상품명_토큰)에서 불필요한 토큰만 걸러내는 함수
def filter_tokens(text):
    words = str(text).split()
    result = []
    for word in words:
        if len(word) < 2:          # 너무 짧은 토큰
            continue
        if word.isdigit():         # 숫자로만 구성된 토큰
            continue
        if word in stopwords:      # 최소한의 불용어
            continue
        result.append(word)
    return " ".join(result)

# 3) 전체 상품명_토큰에 적용해서 상품명_정제 컬럼을 만든다
df["상품명_정제"] = df["상품명_토큰"].apply(filter_tokens)

# 4) 적용 전 / 적용 후를 단어수와 함께 나란히 비교
compare = df[["상품명", "상품명_토큰", "상품명_정제"]].head(10).copy()
compare["적용 전 단어수"] = compare["상품명_토큰"].str.split().str.len()
compare["적용 후 단어수"] = compare["상품명_정제"].str.split().str.len().fillna(0).astype(int)
display(compare)

# 5) 전체 593권 기준 적용 전/후 요약
before_total = df["상품명_토큰"].str.split().str.len().sum()
after_total = df["상품명_정제"].str.split().str.len().fillna(0).sum()
before_empty = int((df["상품명_토큰"] == "").sum())
after_empty = int((df["상품명_정제"] == "").sum())

print("=== 필터링 적용 전/후 요약 (593권 전체) ===")
print(f"전체 단어수   : {before_total}개 -> {int(after_total)}개  (제거 {before_total - int(after_total)}개)")
print(f"빈 문자열 제목 : {before_empty}권 -> {after_empty}권")

,상품명,상품명_토큰,상품명_정제,적용 전 단어수,적용 후 단어수
0,싯다르타,싯다르타,싯다르타,1,1
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판,한국사 이상 현상 연구원 일반판,5,5
2,빵충 사육 준수 사항,빵 충 사육 준수 사항,사육 준수 사항,5,3
3,테오,테오,테오,1,1
4,수족관,수족관,수족관,1,1
5,녹색 절벽의 신자들,녹색 절벽 신자,녹색 절벽 신자,3,3
6,모순,모순,모순,1,1
7,서리꽃 세트,서리 꽃 세트,서리 세트,3,2
8,데미안,데미안,데미안,1,1
9,브람스를 좋아하세요,브람스,브람스,1,1


=== 필터링 적용 전/후 요약 (593권 전체) ===
전체 단어수   : 1869개 -> 1563개  (제거 306개)
빈 문자열 제목 : 7권 -> 21권


## 실습 4 결과 정리

형태소 분석 결과(`상품명_토큰`)에서 최소한의 필터만 적용해 `상품명_정제`를 만들었다.

- 1글자 토큰 제거
- 숫자로만 된 토큰 제거
- 불용어 1개(`에디션`)만 제거 — 분야 핵심 단어(파이썬, 데이터, 경제, 투자 등)는 그대로 둠

### 실행 결과

| 항목 | 적용 전 (상품명_토큰) | 적용 후 (상품명_정제) |
|---|---|---|
| 전체 단어수 (593권 합) | 1,869개 | 1,563개 (-306개) |
| 빈 문자열 제목 | 7권 | 21권 |

### 확인한 것

- `빵 충 사육 준수 사항` -> `사육 준수 사항`처럼, 1글자 명사(`빵`, `충`)가 걸러지면서
  제목이 더 짧고 핵심적인 단어 위주로 정리됐다.
- 그러나 **빈 문자열 제목이 7권에서 21권으로 늘었다.** 1글자 명사만으로 이루어졌던 제목
  14권이 이번 필터링 단계에서 추가로 비어버렸다.
- 필터를 최소한으로 적용했는데도 이런 부작용이 생긴 것은, **전처리를 더한다고 항상 좋아지는
  것은 아니라는 점**을 보여준다.

### 결론

토큰 정제와 불용어 제거는 제목을 더 간결하게 만들었지만, 동시에 **정보를 잃은 제목도 늘렸다.**
이 트레이드오프가 실제 분류 정확도에 어떤 영향을 주는지는 지금 판단할 수 없다.
**실습 5에서 Baseline(47.90%)과 직접 비교해야만** 이 필터링이 도움이 됐는지 알 수 있다.

### 실습 5. Improved 분류 모델 평가하기

 

이제 형태소 분석과 단어 필터링을 적용한 텍스트로 다시 TF-IDF와 Naive Bayes를 학습합니다.

 

알고리즘 자체는 바꾸지 않습니다.

### 실습 6. 추천 결과의 문제 개선하기

이번에는 Chapter 05와 Chapter 06의 추천 기능을 다시 봅니다.
 
기존 추천 흐름은 다음과 같았습니다.

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ============ Baseline: 기존 방식 (Chapter 05) ============
# 전체 도서와 비교, 무조건 Top 5를 채운다 (유사도 0이어도 포함)
def recommend_baseline(selected_index, df, top_n=5):
    vectorizer = TfidfVectorizer()
    matrix = vectorizer.fit_transform(df["상품명"])
    scores = cosine_similarity(matrix[selected_index], matrix).ravel()
    scores[selected_index] = -1  # 자기 자신 제외
    top_idx = scores.argsort()[::-1][:top_n]
    result = df.iloc[top_idx][["상품명", "분야"]].copy()
    result["similarity"] = scores[top_idx].round(4)
    return result

# ============ Improved: 개선 방식 ============
# 개선 1) 같은 분야만 후보로 사용
# 개선 2) 형태소 분석 + 필터링한 텍스트(상품명_정제)로 TF-IDF
# 개선 3) similarity > 0 인 도서만 남기고, 부족하면 억지로 채우지 않는다
def recommend_improved(selected_index, df, top_n=5):
    selected_category = df.loc[selected_index, "분야"]
    candidate_df = df[df["분야"] == selected_category].copy()

    vectorizer = TfidfVectorizer()
    matrix = vectorizer.fit_transform(candidate_df["상품명_정제"])

    local_idx = candidate_df.index.get_loc(selected_index)
    scores = cosine_similarity(matrix[local_idx], matrix).ravel()
    candidate_df["similarity"] = scores

    candidate_df = candidate_df[candidate_df.index != selected_index]
    candidate_df = candidate_df[candidate_df["similarity"] > 0]   # 핵심 조건
    candidate_df = candidate_df.sort_values("similarity", ascending=False).head(top_n)

    return candidate_df[["상품명", "분야", "similarity"]]

# ============ Before / After 확인 ============
sample_title = "소년이 온다"
selected_index = df[df["상품명"] == sample_title].index[0]

print(f"=== 선택 도서: {sample_title} ===")
print()
print("--- Baseline ---")
display(recommend_baseline(selected_index, df))

print()
print("--- Improved ---")
improved_result = recommend_improved(selected_index, df)
if improved_result.empty:
    print("현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다.")
else:
    display(improved_result)

=== 선택 도서: 소년이 온다 ===

--- Baseline ---


,상품명,분야,similarity
384,특이점이 온다,과학,0.4421
296,기획자의 시대가 온다,자기계발,0.3404
193,보랏빛 소가 온다,경제/경영,0.3404
590,이것이 자바다,컴퓨터/IT,0.0000
589,밑바닥부터 시작하는 딥러닝 1(리마스터판),컴퓨터/IT,0.0000



--- Improved ---
현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다.


### 실습 7. Streamlit 앱에서 최종 검증하기

마지막으로 Notebook에서 검증한 개선 방법을 app.py에 연결합니다.

앱은 Chapter 06과 마찬가지로 두 기능만 유지합니다.